# 3.1.4 GBDT+LR：树叶特征与概率校准

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

单独掌握经典两阶段 CTR 模型：用 XGBoost 学条件规则，将叶节点 one-hot 后训练 LR，并复现完整在线推理链。

## Setup

本 Notebook 的默认真实数据是 **GroupLens MovieLens latest-small：经典评分与邻域任务**。`smoke` 档读取仓库内可审计的确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互、曝光、标签或行为序列。切片规则、源地址、哈希与许可记录在 `data/README.md` 及对应数据目录。

**主要资料：** [Practical Lessons from Predicting Clicks on Ads at Facebook](https://research.facebook.com/publications/practical-lessons-from-predicting-clicks-on-ads-at-facebook/)

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("RECSYS_PROFILE", "smoke")
PROFILE = os.environ["RECSYS_PROFILE"]
from recsys_lab.data import (load_movielens, movielens_provenance, load_amazon_2023,
                             amazon_provenance, load_kuairand, kuairand_provenance)
DATASET_KEY = "movielens"
if DATASET_KEY == "movielens":
    real_ratings, real_movies = load_movielens()
    real_interactions = real_ratings
    REAL_DATASET = movielens_provenance(real_ratings)
elif DATASET_KEY == "amazon-2023":
    real_ratings = load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_provenance(real_ratings)
else:
    real_interactions, real_movies = load_kuairand()
    real_ratings = real_interactions
    REAL_DATASET = kuairand_provenance(real_interactions)
print({"profile": PROFILE, "root": str(ROOT), "real_dataset": REAL_DATASET})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'profile': 'smoke', 'root': '/workspace', 'real_dataset': {'dataset': 'MovieLens latest-small (GroupLens, generated 2018-09-26)', 'source': 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', 'license_file': '/workspace/data/ml-latest-small/README.txt', 'rows_used': 26732, 'users_used': 120, 'items_used': 600, 'time_min_utc': '1996-10-17T11:51:49+00:00', 'time_max_utc': '2018-09-13T21:38:16+00:00', 'positive_rule': 'like := observed rating >= 4.0; very_like := observed rating >= 4.5', 'randomly_fabricated_rows': 0}}


## 来源论文与阅读提示

**He et al. (2014), Practical Lessons from Predicting Clicks on Ads at Facebook** 的关键工程判断是把 GBDT 当作监督式特征变换：每棵树的叶节点代表一组条件规则，叶 one-hot 再进入 LR。它不是端到端模型，因此训练/服务必须同时版本化树、叶编码器和 LR。

### 公式怎么读（直觉版）

一棵决策树像连续做选择题：“是否晚间？”“是否移动设备？”最终落到一个叶子。把每棵树落到的叶子变成 0/1 开关，再由 LR 做加权求和。Sigmoid 最后把任意实数分数压到 0～1，变成点击概率；为什么要用 LogLoss 训练，可先看 **3.0 推荐算法数学基础** 的概率图。

In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_squared_error, roc_auc_score, log_loss

SEED = 2026
torch.manual_seed(SEED)
from recsys_lab.data import load_movielens, leave_last_out, movielens_provenance

ratings, movies = load_movielens(max_users=48, max_items=360, min_user_events=12)
train_ratings, test_ratings = leave_last_out(ratings)
n_users, n_items = ratings.user_id.nunique(), ratings.item_id.nunique()

print({
    "rows": len(ratings), "users": ratings.user_id.nunique(), "items": ratings.item_id.nunique(),
    "train_rows": len(train_ratings), "test_rows": len(test_ratings),
    "sparsity": round(1 - len(train_ratings) / (n_users * n_items), 3),
    "source": movielens_provenance(ratings)["source"], "fabricated_rows": 0,
})
display(ratings[["userId", "movieId", "rating", "timestamp", "title", "genres"]].head(8))

{'rows': 10487, 'users': 48, 'items': 360, 'train_rows': 10439, 'test_rows': 48, 'sparsity': 0.396, 'source': 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', 'fabricated_rows': 0}


,userId,movieId,rating,timestamp,title,genres
0,140,318,4.0,942840891,"Shawshank Redemption, The (1994)",Crime|Drama
1,140,527,5.0,942840891,Schindler's List (1993),Drama|War
2,140,1221,3.0,942840891,"Godfather: Part II, The (1974)",Crime|Drama
3,140,50,3.0,942840991,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
4,140,595,5.0,942840991,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX
5,140,1198,4.0,942841170,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure
6,140,2028,5.0,942841170,Saving Private Ryan (1998),Action|Drama|War
7,140,1721,4.0,942841219,Titanic (1997),Drama|Romance


---

## 4. 因子分解机（FM）：为稀疏特征学习二阶交互

### 4.1 为什么 MF 不够？

CTR 排序不只有 `user_id` 和 `item_id`，还会有设备、小时、地域、品类、价格等上下文。直接为每一对特征做 one-hot 交叉会导致参数爆炸；普通 LR 又只能做加法，无法表达“某用户群在晚间更喜欢某品类”。

FM 为每个特征 $i$ 学一个隐向量 $v_i$，用内积共享所有二阶交叉统计：

$$
\hat y(x)=w_0+\sum_i w_ix_i+\sum_{i<j}\langle v_i,v_j\rangle x_ix_j
$$

朴素二阶项是 $O(n^2)$，利用恒等式可化为 $O(nk)$：

$$
\sum_{i<j}\langle v_i,v_j\rangle x_ix_j
=\frac12\sum_f\left[\left(\sum_i v_{if}x_i\right)^2-\sum_i v_{if}^2x_i^2\right]
$$

当特征只有 user one-hot 和 item one-hot 时，FM 的交叉项就退化为 MF；因此 FM 可以理解为 MF 对通用稀疏特征的扩展。

### 4.2 数据：从真实评分构造可解释的二分类排序任务

In [3]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ctr = ratings.sort_values("timestamp").tail(5000).copy().reset_index(drop=True)
# `click` 是任务别名；标签由真实评分 rating >= 4.0 确定，不使用随机采样。
ctr["click"] = ctr["like"].astype(int)
split = int(len(ctr) * .8)  # 严格按真实 timestamp 切分
categorical = ["user_id", "item_id", "genre_id", "hour", "decade_id"]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_sparse = encoder.fit_transform(ctr.loc[:split-1, categorical])
test_sparse = encoder.transform(ctr.loc[split:, categorical])
scaler = StandardScaler()
train_numeric = scaler.fit_transform(ctr.loc[:split-1, ["item_popularity", "user_activity"]])
test_numeric = scaler.transform(ctr.loc[split:, ["item_popularity", "user_activity"]])
fm_train_x = np.c_[train_sparse, train_numeric].astype("float32")
fm_test_x = np.c_[test_sparse, test_numeric].astype("float32")
ctr_train_y = ctr.loc[:split-1, "click"].to_numpy()
ctr_test_y = ctr.loc[split:, "click"].to_numpy()
print({"train": fm_train_x.shape, "test": fm_test_x.shape, "train_positive_rate": round(ctr_train_y.mean(), 3)})

{'train': (4000, 437), 'test': (1000, 437), 'train_positive_rate': np.float64(0.474)}


---

## 5. GBDT+LR：树负责发现规则，LR 负责组合与校准

### 5.1 核心思想

Facebook 经典 CTR 工作把 GBDT 的每棵树视为一个自动特征生成器。样本落到某棵树的哪个叶子，代表它满足了一组条件，例如：

> `hour > 18 AND device = mobile AND price < 20`

将每棵树的叶节点编号 one-hot 后输入 LR：

$$
P(y=1\mid z)=\sigma(w_0+w^\top z)
$$

这里 $z$ 是所有树叶节点的稀疏指示向量。GBDT 学非线性分桶与条件组合；LR 学各条规则的稳定权重并输出概率。注意这通常是**两阶段训练**，不是端到端联合优化。

### 5.2 训练阶段一：XGBoost 学习树规则

In [4]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

tree_features = ["user_id", "item_id", "genre_id", "hour", "decade_id", "item_popularity", "user_activity"]
# GBDT 使用与 FM 相同的 one-hot 类别语义，避免把类别 ID 错当连续数值。
gbdt_train_x = fm_train_x
gbdt_test_x = fm_test_x

gbdt = XGBClassifier(
    n_estimators=80, max_depth=4, learning_rate=.05, subsample=.9,
    colsample_bytree=.9, eval_metric="logloss", random_state=SEED, n_jobs=1
)
gbdt.fit(gbdt_train_x, ctr_train_y)
train_leaf = gbdt.apply(gbdt_train_x)
test_leaf = gbdt.apply(gbdt_test_x)
print({"trees": train_leaf.shape[1], "train_leaf_matrix": train_leaf.shape})

{'trees': 80, 'train_leaf_matrix': (4000, 80)}


### 5.3 训练阶段二：叶节点 one-hot 后训练 LR

In [5]:
leaf_encoder = OneHotEncoder(handle_unknown="ignore")
train_leaf_onehot = leaf_encoder.fit_transform(train_leaf)
test_leaf_onehot = leaf_encoder.transform(test_leaf)

leaf_lr = LogisticRegression(max_iter=500, C=.7)
leaf_lr.fit(train_leaf_onehot, ctr_train_y)
gbdt_lr_probability = leaf_lr.predict_proba(test_leaf_onehot)[:, 1]

gbdt_lr_auc = float(roc_auc_score(ctr_test_y, gbdt_lr_probability))
gbdt_lr_logloss = float(log_loss(ctr_test_y, gbdt_lr_probability))
print({"leaf_onehot_dim": train_leaf_onehot.shape[1], "GBDT+LR AUC": round(gbdt_lr_auc, 4), "GBDT+LR LogLoss": round(gbdt_lr_logloss, 4)})

{'leaf_onehot_dim': 1049, 'GBDT+LR AUC': 0.6376, 'GBDT+LR LogLoss': 0.7414}


### 5.4 推理：同一条样本必须依次经过两阶段

In [6]:
def predict_gbdt_lr(frame):
    sparse_part = encoder.transform(frame[categorical])
    numeric_part = scaler.transform(frame[["item_popularity", "user_activity"]])
    encoded_features = np.c_[sparse_part, numeric_part].astype("float32")
    leaf = gbdt.apply(encoded_features)
    leaf_onehot = leaf_encoder.transform(leaf)
    return leaf_lr.predict_proba(leaf_onehot)[:, 1]

online_batch = ctr.loc[split:split+4].copy()
online_batch["p_click"] = predict_gbdt_lr(online_batch)
display(online_batch[tree_features + ["click", "p_click"]])

,user_id,item_id,genre_id,hour,decade_id,item_popularity,user_activity,click,p_click
4000,40,210,8,1,6,30.0,218.0,0,0.292489
4001,40,261,5,1,7,26.0,218.0,1,0.831249
4002,40,358,0,1,7,27.0,218.0,0,0.306214
4003,40,111,4,1,4,27.0,218.0,0,0.451394
4004,40,303,7,1,7,28.0,218.0,0,0.279590


### 5.5 结果讨论与边界

**优点**：连续变量无需手工分桶；树能发现条件组合；LR 服务成熟、输出易校准；对中等规模表格特征很强。  
**缺点**：两阶段可能失配；叶节点空间随树数膨胀；高基数 ID 容易过拟合；树规则会随分布漂移而陈旧；难以表达长行为序列。  
**工业注意**：训练与服务必须共享完全相同的树版本、叶编码器和缺失值处理；监控 unseen leaf、特征漂移与概率校准。

与 FM 的关键区别：FM 假设交互可由低秩内积共享；GBDT+LR 假设有效模式可由一组树规则离散化。两者没有绝对胜负，取决于稀疏度、特征类型、数据量和延迟预算。

## Checks

确认树数、叶特征维数、概率范围和 AUC；推理必须复用同一编码器、树模型与 LR。

In [7]:
assert train_leaf.shape[1] == gbdt.n_estimators
assert train_leaf_onehot.shape[1] > train_leaf.shape[1]
assert np.all((gbdt_lr_probability >= 0) & (gbdt_lr_probability <= 1))
assert gbdt_lr_auc > .55
print('PASS：GBDT、叶编码、LR 与在线推理链均有效。')

PASS：GBDT、叶编码、LR 与在线推理链均有效。


In [8]:
result_dir = ROOT / "results" / "chapter_3_1"; result_dir.mkdir(parents=True, exist_ok=True)
payload = {"records": [{"algorithm":"GBDT+LR", "task":"CTR", "metric":"AUC ↑", "value":gbdt_lr_auc, "secondary_metric":"LogLoss ↓", "secondary_value":gbdt_lr_logloss, "online_inference":"树叶编码 → LR 概率", "source_notebook":"3_1_4_gbdt_lr"}]}
(result_dir / "gbdt_lr.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("已写出章节指标：gbdt_lr.json")

已写出章节指标：gbdt_lr.json


## Next Steps

与原始特征 LR、FM 对照；加入概率校准曲线；模拟数据漂移并监控叶节点覆盖变化。